# ⚽ FIFA World Cup 2026 Prediction Model

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

- Talk about my current model and what my model tries to accomplish

- Talk about Datacamp competition, and how this version differs from competition submission.
  


## Model Architecture

- Describe the different variables, modifiers etc and why I chose the values I chose (rho, the multipliers etc.)

In [181]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from collections import Counter

def load_csv(filename : str):

    """
    Loads a CSV from the data/ folder and prints its shape.
    """
    
    df = pd.read_csv(f'data/{filename}')
    print(f"Loaded {filename} with shape {df.shape}")
    return df

group_fixtures = load_csv('group_fixtures.csv')
knockout_slots = load_csv('knockout_slots.csv')
knockout_slots = knockout_slots.drop(columns=['multiplier'])  # Leftover from DataCamp competition

display(group_fixtures)
display(knockout_slots)

Loaded group_fixtures.csv with shape (72, 6)
Loaded knockout_slots.csv with shape (32, 7)


,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"
...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia"
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami"
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta"
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City"


,match_id,round,date_utc,venue,slot_home,slot_away
0,73,Round of 32,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


In [182]:
# Confirmed qualified teams.

confirmed_teams = {
    'FIFA Playoff 1' : 'DR Congo',
    'FIFA Playoff 2' : 'Iraq',
    'UEFA Playoff A' : 'Bosnia and Herzegovina',
    'UEFA Playoff B' : 'Sweden',
    'UEFA Playoff C' : 'Turkey',
    'UEFA Playoff D' : 'Czech Republic',
}

group_fixtures = group_fixtures.replace(confirmed_teams)

In [183]:
# Dictionary of tournament weights, based (loosely) on official FIFA elo calculation data.

FRIENDLY = 7.5
NATIONS_LEAGUE = 20
CONF_QUAL = 25
CONF = 37.5
WC_QUAL = 25
WC = 55
DEFAULT = 10

tournament_weights = {
    'Friendly': FRIENDLY,
    'CONCACAF Nations League': NATIONS_LEAGUE,
    'UEFA Nations League': NATIONS_LEAGUE,
    'UEFA Euro qualification': CONF_QUAL,
    'UEFA Euro': CONF,
    'Copa América qualification': CONF_QUAL,
    'Copa América': CONF,
    'African Cup of Nations qualification': CONF_QUAL,
    'African Cup of Nations': CONF,
    'AFC Asian Cup qualification': CONF_QUAL,
    'AFC Asian Cup': CONF,
    'Gold Cup qualification': CONF_QUAL,
    'Gold Cup': CONF,
    'Oceania Nations Cup qualification': CONF_QUAL,
    'Oceania Nations Cup': CONF,
    'FIFA World Cup qualification': WC_QUAL,
    'FIFA World Cup': WC,
}

In [184]:
def time_decay(df, decay_rate = 0.3, reference_date = pd.Timestamp('2026-06-01')):

    """
    Creates a decay formula that weighs match importance based on how long ago it was played.
    """
    
    df['years_ago'] = np.floor((reference_date - pd.to_datetime(df['date'])).dt.days / 365.25)
    
    df['decay'] = np.exp(-decay_rate * df['years_ago'])
    df['base_adjusted_weight'] = df['decay'] * df['weight']
    
    return df

In [185]:
def performance_weight(df, p = 5.8, q = 1.5):
    
    df['offensive_weight'] = df['base_adjusted_weight'] * (df['opponent_strength'] ** p)
    df['defensive_weight'] = df['base_adjusted_weight'] * (df['opponent_strength'] ** q)
    
    return df

In [186]:
# Updated team names for consistency.

changed_team_names = {
    'Cape Verde' : 'Cabo Verde',
    'Ivory Coast' : "Côte d'Ivoire",
    'United States' : 'USA',
    'Czechia' : 'Czech Republic'
}

In [187]:
def build_team_match_data():
    """
    Creates a DataFrame that shows each match twice: the original match, and one with the home and away times mirrored. This allows me to calculate each team's stats by grouping by 'team', rather than home and away separately. 

    Additionally, it also showcases each team's average elo between 2021-06-01 and 2026-06-01, which will be used to process strength calculations.
    """

    # This is the dataset we'll be working from. Only games from 2021-06-01 to 2026-06-01 will be considered when performing team strength calculations, as data collected before that would not be representative of current team strength.

    results = load_csv('results.csv')

    recent_results = results[(results['date'] >= '2021-06-01') & (results['home_score'].notna())].copy() 
    recent_results['weight'] = recent_results['tournament'].map(
        lambda t: tournament_weights.get(t, DEFAULT)
    )
    home_perspective = recent_results[['date', 'home_team', 'away_team', 'home_score', 'away_score','tournament', 'weight']].copy()
    home_perspective.columns = ['date', 'team', 'opponent', 'goals_scored', 'goals_conceded','tournament', 'weight']
    
    away_perspective = recent_results[['date', 'away_team', 'home_team', 'away_score', 'home_score','tournament', 'weight']].copy()
    away_perspective.columns = ['date', 'team', 'opponent', 'goals_scored', 'goals_conceded','tournament', 'weight']
    
    df = pd.concat([home_perspective, away_perspective], ignore_index = True).replace(changed_team_names)

    # Loads almost every team's average elo between 2021-06-01 abd 2026-06-01. Teams with under 15 games played were excluded from the dataset.

    historical_elo = pd.read_csv('data/wf_avg_elo.csv', index_col = 'team')
    
    teams_with_elo = set(historical_elo.index)
    df = df[df['team'].isin(teams_with_elo) & df['opponent'].isin(teams_with_elo)]

    df['team_avg_elo'] = df['team'].map(historical_elo['avg_elo'])
    df['opponent_avg_elo'] = df['opponent'].map(historical_elo['avg_elo'])

    global_average_elo = historical_elo['avg_elo'].mean()
    df['team_strength'] = df['team_avg_elo'] / global_average_elo
    df['opponent_strength'] = df['opponent_avg_elo'] / global_average_elo

    df = time_decay(df)
    df = performance_weight(df)

    return df

team_match_data = build_team_match_data()

Loaded results.csv with shape (49353, 9)


In [188]:
def calc_avg_weighted_goals(df):

    """
    A formula that calculates the weighted average of goals socred per game, based on decay, tournament weight and opponent strength.
    """
    
    sum_offensive_weight = df.groupby('team')['offensive_weight'].sum()
    df['weighted_goals'] = df['goals_scored'] * df['offensive_weight']
    sum_weighted_goals = df.groupby('team')['weighted_goals'].sum()
    
    avg_weighted_goals = sum_weighted_goals / sum_offensive_weight
    avg_weighted_goals.name = 'avg_weighted_goals'

    return avg_weighted_goals

avg_weighted_goals = calc_avg_weighted_goals(team_match_data)

In [189]:
def calc_avg_weighted_conceded(df):
    
    sum_defensive_weight = df.groupby('team')['defensive_weight'].sum()
    
    df['weighted_conceded'] = df['goals_conceded'] * df['defensive_weight']
    sum_weighted_conceded = df.groupby('team')['weighted_conceded'].sum()
    
    avg_weighted_conceded = sum_weighted_conceded / sum_defensive_weight
    avg_weighted_conceded.name = 'avg_weighted_conceded'

    return avg_weighted_conceded

avg_weighted_conceded = calc_avg_weighted_conceded(team_match_data)

In [190]:
def build_wc_team_stats(avg_weighted_goals, avg_weighted_conceded):
    
    team_stats = pd.DataFrame({
        'avg_weighted_goals' : avg_weighted_goals,
        'avg_weighted_conceded' : avg_weighted_conceded,
    })

    #A filter that removes all non-WC teams from team_stats

    wc_teams = set(group_fixtures['home_team']) | set(group_fixtures['away_team'])
    wc_team_stats = team_stats[team_stats.index.isin(wc_teams)]

    wf_elo = load_csv('elo_ratings_wc2026.csv')
    wf_elo['country'] = wf_elo['country'].replace(changed_team_names)
    wc_current_elo = wf_elo[wf_elo['snapshot_date'] == '2026-05-27'].set_index('country')
    wc_current_elo = wc_current_elo.rename(columns = {
        'rating' : 'current_elo'
    })

    wc_team_stats['current_elo'] = wc_current_elo['current_elo']

    return wc_team_stats

wc_team_stats = build_wc_team_stats(avg_weighted_goals, avg_weighted_conceded)
wc_team_stats.sort_values('current_elo', ascending=False)
wc_team_stats

Loaded elo_ratings_wc2026.csv with shape (4683, 23)


,avg_weighted_goals,avg_weighted_conceded,current_elo
team,,,
Algeria,1.660365,0.752000,1743
Argentina,1.712194,0.563877,2113
Australia,1.129446,0.869466,1783
Austria,1.426586,1.125702,1827
Belgium,1.341567,0.979520,1867
Bosnia and Herzegovina,0.860834,1.764945,1594
Brazil,1.430140,0.759429,1984
Cabo Verde,1.108857,1.040669,1549
Canada,1.040269,1.053830,1784


In [191]:
host_teams = {'USA', 'Mexico', 'Canada'}

venue_country = {
    'Estadio Azteca, Mexico City' : 'Mexico',
    'Estadio Akron, Guadalajara' : 'Mexico',
    'Estadio BBVA, Monterrey' : 'Mexico',
    'BMO Field, Toronto' : 'Canada',
    'BC Place, Vancouver' : 'Canada',
    'SoFi Stadium, Los Angeles' : 'USA',
    "Levi's Stadium, Santa Clara" : 'USA',
    'MetLife Stadium, East Rutherford' : 'USA',
    'Gillette Stadium, Boston' : 'USA',
    'NRG Stadium, Houston' : 'USA',
    'AT&T Stadium, Dallas' : 'USA',
    'Lincoln Financial Field, Philadelphia' : 'USA',
    'Mercedes-Benz Stadium, Atlanta' : 'USA',
    'Lumen Field, Seattle' : 'USA',
    'GEHA Field at Arrowhead Stadium, Kansas City' : 'USA',
    'Hard Rock Stadium, Miami' : 'USA',
}

In [192]:
class MatchSimulator:
    
    def __init__(self, team_stats):

        self.team_stats = team_stats

    def elo_modifier(self, team_elo, opponent_elo, n = 2):
        
        """
        Calcualates the elo_modifier, which determines how significant a team's elo is heighted in head to head matches.
        """
    
        return (team_elo / opponent_elo) ** n

    def host_boost(self, team, venue):
        
        """
        Returns the additive xG boost a team receives for playing at home. Full boost (0.20) if the team is playing in its own country, reduced boost (0.10) if playing in a co-host country, 0 otherwise.
        """
    
        if team not in host_teams:
            return 0.0
        host_country = venue_country.get(venue)
        if team == host_country:
            return 0.20
            
        return 0.10

    def expected_goals(self, home_team, away_team, venue = None):

        home_elo = self.team_stats.loc[home_team, 'current_elo']
        away_elo = self.team_stats.loc[away_team, 'current_elo']

        home_modifier = self.elo_modifier(home_elo, away_elo)
        away_modifier = self.elo_modifier(away_elo, home_elo)
    
        home_attack  = self.team_stats.loc[home_team, 'avg_weighted_goals']
        away_defense = self.team_stats.loc[away_team, 'avg_weighted_conceded']
        away_attack  = self.team_stats.loc[away_team, 'avg_weighted_goals']
        home_defense = self.team_stats.loc[home_team, 'avg_weighted_conceded']
    
        x_home = (home_attack * away_defense) * home_modifier
        x_away = (away_attack * home_defense) * away_modifier

        x_home += self.host_boost(home_team, venue)
        x_away += self.host_boost(away_team, venue)
        
        return x_home, x_away

    def rho_correction(self, home_goals, away_goals, x_home, x_away, rho = -0.13): 

        """
        The Dixon-Coles rho correction function, used to increase the frequency of 1 - 1, 1 - 0, 0 - 0 and 0 - 1 scorelines, as they tend to be underrepresented in poisson regression.
        """
    
        if home_goals == 0 and away_goals == 0:
            return 1 - (x_home * x_away * rho)
        elif home_goals == 0 and away_goals == 1:
            return 1 + (x_home * rho)
        elif home_goals == 1 and away_goals == 0:
            return 1 + (x_away * rho)
        elif home_goals == 1 and away_goals == 1:
            return 1 - rho
        else:
            return 1

    def historical_avg_cards(self):
        
        bookings = load_csv('wc_bookings.csv')
        recent = bookings[bookings['tournament_id'].isin(['WC-2014', 'WC-2018', 'WC-2022'])]
        match_cards = recent.groupby('match_id').agg(
            yellows=('yellow_card', 'sum'),
            reds=('red_card', 'sum')
        ).reset_index()

        self.avg_yellow = match_cards['yellows'].mean() / 2 # For average cards per team per game
        self.avg_red = match_cards['reds'].mean() / 2

    def simulate_regular_time(self, x_home, x_away):
    
        home_poss = np.arange(0,8)
        away_poss = np.arange(0,8)
    
        home_prob = stats.poisson.pmf(home_poss, x_home)
        away_prob = stats.poisson.pmf(away_poss, x_away) #Calculates poisson possibility of each scoreline
        score_matrix = np.outer(home_prob, away_prob) #multiplies every element of home_poss against away_poss
    
        score_matrix[1][0] *= self.rho_correction(1, 0, x_home, x_away)
        score_matrix[0][1] *= self.rho_correction(0, 1, x_home, x_away)
        score_matrix[0][0] *= self.rho_correction(0, 0, x_home, x_away)
        score_matrix[1][1] *= self.rho_correction(1, 1, x_home, x_away)

        flat_score_matrix = score_matrix.flatten()
        flat_score_matrix = flat_score_matrix / flat_score_matrix.sum()
        i = np.random.choice(len(flat_score_matrix), p = flat_score_matrix)

        home_goals = i // 8
        away_goals = i % 8
        return home_goals, away_goals

    def simulate_extra_time(self, home_team, away_team, venue = None):

        x_home, x_away = self.expected_goals(home_team, away_team, venue)
        x_home, x_away = x_home / 3, x_away / 3

        x_yellow, x_red = self.avg_yellow, self.avg_red
        x_yellow, x_red = x_yellow / 3, x_red/ 3
        
        et_home, et_away = self.simulate_regular_time(x_home, x_away)
        et_yellow = np.random.poisson(x_yellow)
        et_red = np.random.poisson(x_red)

        return et_home, et_away, et_yellow, et_red

    def monte_carlo(self, home_team, away_team, simulations = 10000, venue = None):

        scores = []
        x_home, x_away = self.expected_goals(home_team, away_team, venue)

        for _ in range(simulations):
            home_goals, away_goals = self.simulate_regular_time(x_home, x_away)
            scores.append((home_goals, away_goals))

        home_wins = sum(h > a for h, a in scores)
        away_wins = sum(a > h for h, a in scores)
        draws = sum(h == a for h, a in scores)
                
        return scores, home_wins, away_wins, draws

    def monte_carlo_gs(self, home_team, away_team, simulations = 10000, threshold = 0.29, venue = None):

        scores, home_wins, away_wins, draws = self.monte_carlo(home_team, away_team, simulations, venue)
        total_results = home_wins + away_wins + draws

        home_yellow = np.random.poisson(self.avg_yellow)
        home_red    = np.random.poisson(self.avg_red)
        away_yellow = np.random.poisson(self.avg_yellow)
        away_red    = np.random.poisson(self.avg_red)
    
        if draws / total_results > threshold:
            winning_team = 'draw'
            filtered_scores = [(h, a) for h, a in scores if h == a]
            draw_goals = round(np.mean([h for h, a in filtered_scores]))
            home_mode, away_mode = draw_goals, draw_goals
        elif home_wins > away_wins:
            winning_team = 'home'
            filtered_scores = [(h, a) for h, a in scores if h > a]
            home_mode, away_mode = Counter(filtered_scores).most_common(1)[0][0]
        else:
            winning_team = 'away'
            filtered_scores = [(h, a) for h, a in scores if h < a]
            home_mode, away_mode = Counter(filtered_scores).most_common(1)[0][0]
            
        return home_mode, away_mode, winning_team, home_yellow, home_red, away_yellow, away_red


    def monte_carlo_ko(self, home_team, away_team, simulations = 10000, threshold = 0.28, venue = None):
    
        scores, home_wins, away_wins, draws = self.monte_carlo(home_team, away_team, simulations, venue)
        total_results = home_wins + away_wins + draws

        home_yellow = np.random.poisson(self.avg_yellow)
        home_red    = np.random.poisson(self.avg_red)
        away_yellow = np.random.poisson(self.avg_yellow)
        away_red    = np.random.poisson(self.avg_red)
    
        if draws / total_results > threshold:
            et_home, et_away, et_yellow, et_red = self.simulate_extra_time(home_team, away_team, venue)
            filtered_scores = [(h, a) for h, a in scores if h == a]
            draw_goals = round(np.mean([h for h, a in filtered_scores]))
            
            home_mode = draw_goals + et_home
            away_mode = draw_goals + et_away
            home_yellow += et_yellow 
            home_red += et_red
            away_yellow += et_yellow
            away_red += et_red
        
            if et_home > et_away:
                winning_team = 'home'
            elif et_away > et_home:
                winning_team = 'away'
            else:
                winning_team = 'home' if np.random.random() > 0.5 else 'away'
            
        elif home_wins > away_wins:
            winning_team = 'home'
            filtered_scores = [(h, a) for h, a in scores if h > a]
            home_mode, away_mode = Counter(filtered_scores).most_common(1)[0][0]
        else:
            winning_team = 'away'
            filtered_scores = [(h, a) for h, a in scores if h < a]
            home_mode, away_mode = Counter(filtered_scores).most_common(1)[0][0]
        
        return home_mode, away_mode, winning_team, home_yellow, home_red, away_yellow, away_red

match_sim = MatchSimulator(wc_team_stats)
match_sim.historical_avg_cards()

Loaded wc_bookings.csv with shape (3178, 26)


In [193]:
class TournamentSimulator:
    
    def __init__(self, match_sim, group_fixtures, knockout_slots):
        self.match_sim = match_sim
        self.group_fixtures = group_fixtures
        self.knockout_slots = knockout_slots

    def group_results(self, predictions):
        for i, match in self.group_fixtures.iterrows():
            home_goals, away_goals, winner, home_yellow, home_red, away_yellow, away_red = self.match_sim.monte_carlo_gs(match['home_team'], match['away_team'], simulations = 10000, venue = match['venue'])

            predictions.loc[i, 'predicted_home_goals'] = home_goals
            predictions.loc[i, 'predicted_away_goals'] = away_goals
            predictions.loc[i, 'home_yellow_cards'] = home_yellow
            predictions.loc[i, 'home_red_cards']    = home_red
            predictions.loc[i, 'away_yellow_cards'] = away_yellow
            predictions.loc[i, 'away_red_cards']    = away_red
            predictions.loc[i, 'winning_team']         = winner
            

        standings = self.group_standings()
        self.group_rankings = self.build_group_rankings(standings)
        self.qualified_thirds = self.best_third(standings).head(8)

        return predictions

    def group_standings(self):

        stats = {}

        for group_name in sorted(group_fixtures['group'].unique()):
            current_group = group_fixtures[group_fixtures['group'] == group_name]

            home_teams = set(current_group['home_team'])
            away_teams = set(current_group['away_team'])

            teams = sorted(home_teams | away_teams)

            for team in teams:
                stats[team] = {
                    'Pts' : 0,
                    'GD' : 0,
                    'GF' : 0,
                    'GA' : 0
                }

        for i, match in group_predictions.iterrows():

            hg = match['predicted_home_goals']
            ag = match['predicted_away_goals']
            home = match['home_team']
            away = match['away_team']

            stats[home]['GF'] += hg
            stats[home]['GA'] += ag
            stats[away]['GF'] += ag
            stats[away]['GA'] += hg
            stats[home]['GD'] = stats[home]['GF'] - stats[home]['GA']
            stats[away]['GD'] = stats[away]['GF'] - stats[away]['GA']
            
            if match['winning_team'] == 'home':
                stats[home]['Pts'] += 3
            elif match['winning_team'] == 'away':
                stats[away]['Pts'] += 3
            else:
                stats[home]['Pts'] += 1
                stats[away]['Pts'] += 1

        standings = pd.DataFrame(stats).T
    
        home_map = group_fixtures.set_index('home_team')['group']
        away_map = group_fixtures.set_index('away_team')['group']
    
        team_groups = pd.concat([home_map, away_map])
        team_groups = team_groups[~team_groups.index.duplicated(keep = 'first')]
        standings['group'] = team_groups
        standings = standings.sort_values(['group', 'Pts', 'GD', 'GF'], ascending=[True, False, False, False])

        return standings

    def build_group_rankings(self, standings):

        group_rankings = {} #stores the predicted team order for each group

        for group_name in sorted(self.group_fixtures["group"].unique()):
            teams = standings[standings['group'] == group_name].index.tolist()
            group_rankings[group_name] = teams #stores the teams into the group_rankings variable

        return group_rankings

    def best_third(self, standings):

        third_place_teams = []
    
        for group_name, teams in self.build_group_rankings(standings).items():
            third_place_teams.append(teams[2])

        third_place_order = standings[standings.index.isin(third_place_teams)].sort_values(['Pts', 'GD', 'GF'], ascending = [False, False, False])

        return third_place_order

    def get_team_slot(self, slot):

        """
        Finds the specific slot a team occupies.
        """
    
        words = slot.split()

        if slot.startswith("Winner"):
            position = 0
            group_name = words[-1]
        
        elif slot.startswith("Runner-up"):
            position = 1
            group_name = words[-1]
        
        elif slot.startswith("Best 3rd"):
            return self.third_assignments[slot]

        return self.group_rankings[group_name][position]

    def assign_best_thirds(self):
        
        """Maps each qualifying group's 3rd-place team to its FIFA-assigned R32 opponent slot, using the official 2026 third-place allocation table.
        """
        
        import json
        with open('data/third_place_table.json') as f:
            third_place_table = json.load(f)

        qualified_groups = sorted(self.qualified_thirds['group'].tolist())
        key = ''.join(qualified_groups)
        row = third_place_table[key]

        self.third_assignments = {}
        for fixed_slot, third_group_code in row.items():
            third_group = third_group_code[-1]
            team = self.qualified_thirds[self.qualified_thirds['group'] == third_group].index[0]
            self.third_assignments[fixed_slot] = team

    def run_knockout_stages(self, predictions):
        
        match_winners = {}
        match_losers = {}
        self.assign_best_thirds()

        for i, slot in self.knockout_slots.iterrows():
            match_id = slot['match_id']

            if slot["slot_home"].startswith(("Winner Match", "Loser Match")):
                home_team = self.resolve_match_slot(slot["slot_home"], match_winners, match_losers)
            else:
                home_team = self.get_team_slot(slot["slot_home"])

            if slot["slot_away"].startswith("Best 3rd"):
                group_letter = slot["slot_home"].split()[-1]
                away_team = self.third_assignments['1' + group_letter]
            elif slot["slot_away"].startswith(("Winner Match", "Loser Match")):
                away_team = self.resolve_match_slot(slot["slot_away"], match_winners, match_losers)
            else:
                away_team = self.get_team_slot(slot["slot_away"])

            

            home_goals, away_goals, winning_team, home_yellow, home_red, away_yellow, away_red = self.match_sim.monte_carlo_ko(home_team, away_team, venue = slot['venue'])

            winner = home_team if winning_team == 'home' else away_team
            loser  = away_team if winning_team == 'home' else home_team
            match_winners[match_id] = winner
            match_losers[match_id]  = loser

            predictions.loc[i, 'predicted_home_team']  = home_team
            predictions.loc[i, 'predicted_away_team']  = away_team
            predictions.loc[i, 'predicted_home_goals'] = home_goals
            predictions.loc[i, 'predicted_away_goals'] = away_goals
            predictions.loc[i, 'home_yellow_cards'] = home_yellow
            predictions.loc[i, 'home_red_cards']    = home_red
            predictions.loc[i, 'away_yellow_cards'] = away_yellow
            predictions.loc[i, 'away_red_cards']    = away_red
            predictions.loc[i, 'match_winner']         = winner
            predictions.loc[i, 'penalties'] = (home_goals == away_goals)

        return predictions

    def resolve_match_slot(self, slot, match_winners, match_losers):
        
        """
        Resolves slots like 'Winner Match 73' or 'Loser Match 101'
        using results from earlier knockout matches.
        """
        
        match_id = int(slot.split()[-1])
        if slot.startswith("Winner"):
            return match_winners[match_id]
        else:
            return match_losers[match_id]

tournament_sim = TournamentSimulator(match_sim, group_fixtures, knockout_slots)

In [162]:
group_predictions = group_fixtures.copy()
group_predictions = tournament_sim.group_results(group_predictions)
group_predictions

In [163]:
standings = tournament_sim.group_standings()

for group_name in sorted(standings['group'].unique()):
    display(standings[standings['group'] == group_name])

,Pts,GD,GF,GA,group
Mexico,7.0,3.0,4.0,1.0,A
South Korea,7.0,2.0,4.0,2.0,A
South Africa,1.0,-2.0,1.0,3.0,A
Czech Republic,1.0,-3.0,2.0,5.0,A


,Pts,GD,GF,GA,group
Switzerland,9.0,8.0,9.0,1.0,B
Canada,6.0,3.0,5.0,2.0,B
Bosnia and Herzegovina,3.0,-4.0,2.0,6.0,B
Qatar,0.0,-7.0,1.0,8.0,B


,Pts,GD,GF,GA,group
Brazil,7.0,5.0,6.0,1.0,C
Morocco,7.0,4.0,5.0,1.0,C
Scotland,3.0,-3.0,2.0,5.0,C
Haiti,0.0,-6.0,1.0,7.0,C


,Pts,GD,GF,GA,group
USA,7.0,2.0,4.0,2.0,D
Australia,5.0,1.0,4.0,3.0,D
Turkey,3.0,-1.0,4.0,5.0,D
Paraguay,1.0,-2.0,2.0,4.0,D


,Pts,GD,GF,GA,group
Germany,7.0,4.0,6.0,2.0,E
Ecuador,5.0,1.0,2.0,1.0,E
Côte d'Ivoire,4.0,1.0,3.0,2.0,E
Curaçao,0.0,-6.0,0.0,6.0,E


...

,Pts,GD,GF,GA,group
Spain,9.0,10.0,10.0,0.0,H
Uruguay,6.0,0.0,2.0,2.0,H
Cabo Verde,1.0,-5.0,1.0,6.0,H
Saudi Arabia,1.0,-5.0,1.0,6.0,H


,Pts,GD,GF,GA,group
France,9.0,4.0,5.0,1.0,I
Norway,6.0,2.0,5.0,3.0,I
Senegal,3.0,-1.0,2.0,3.0,I
Iraq,0.0,-5.0,0.0,5.0,I


,Pts,GD,GF,GA,group
Argentina,9.0,6.0,6.0,0.0,J
Algeria,6.0,0.0,4.0,4.0,J
Austria,3.0,-2.0,3.0,5.0,J
Jordan,0.0,-4.0,2.0,6.0,J


,Pts,GD,GF,GA,group
Portugal,9.0,5.0,6.0,1.0,K
Colombia,2.0,-1.0,3.0,4.0,K
DR Congo,2.0,-2.0,1.0,3.0,K
Uzbekistan,2.0,-2.0,1.0,3.0,K


,Pts,GD,GF,GA,group
England,9.0,8.0,8.0,0.0,L
Croatia,6.0,1.0,4.0,3.0,L
Panama,3.0,-2.0,3.0,5.0,L
Ghana,0.0,-7.0,1.0,8.0,L


In [164]:
tournament_sim.qualified_thirds 

,Pts,GD,GF,GA,group
Côte d'Ivoire,4.0,1.0,3.0,2.0,E
Egypt,3.0,0.0,3.0,3.0,G
Turkey,3.0,-1.0,4.0,5.0,D
Senegal,3.0,-1.0,2.0,3.0,I
Austria,3.0,-2.0,3.0,5.0,J
Panama,3.0,-2.0,3.0,5.0,L
Scotland,3.0,-3.0,2.0,5.0,C
Bosnia and Herzegovina,3.0,-4.0,2.0,6.0,B


In [165]:
knockout_predictions = knockout_slots.copy()
tournament_sim.run_knockout_stages(knockout_predictions)
knockout_predictions

,match_id,round,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,home_yellow_cards,home_red_cards,away_yellow_cards,away_red_cards,match_winner,penalties
0,73,Round of 32,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Canada,1.0,2.0,4.0,0.0,1.0,0.0,Canada,False
1,74,Round of 32,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Netherlands,2.0,2.0,0.0,0.0,2.0,0.0,Netherlands,True
2,75,Round of 32,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Scotland,3.0,1.0,0.0,1.0,1.0,0.0,Germany,False
3,76,Round of 32,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Japan,Morocco,1.0,1.0,1.0,0.0,2.0,0.0,Japan,True
4,77,Round of 32,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Ecuador,Norway,1.0,1.0,2.0,0.0,4.0,0.0,Ecuador,True
5,78,Round of 32,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Turkey,3.0,1.0,1.0,0.0,1.0,0.0,France,False
6,79,Round of 32,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Côte d'Ivoire,1.0,2.0,4.0,0.0,3.0,0.0,Côte d'Ivoire,False
7,80,Round of 32,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Senegal,1.0,0.0,1.0,0.0,1.0,0.0,England,False
8,81,Round of 32,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Austria,2.0,1.0,1.0,0.0,3.0,0.0,Belgium,False
9,82,Round of 32,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),USA,Bosnia and Herzegovina,2.0,0.0,1.0,0.0,1.0,0.0,USA,False


In [166]:
for round_name in knockout_predictions['round'].unique():
    print(round_name)
    display(knockout_predictions[knockout_predictions['round'] == round_name])

Round of 32


,match_id,round,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,home_yellow_cards,home_red_cards,away_yellow_cards,away_red_cards,match_winner,penalties
0,73,Round of 32,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Canada,1.0,2.0,4.0,0.0,1.0,0.0,Canada,False
1,74,Round of 32,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Netherlands,2.0,2.0,0.0,0.0,2.0,0.0,Netherlands,True
2,75,Round of 32,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Scotland,3.0,1.0,0.0,1.0,1.0,0.0,Germany,False
3,76,Round of 32,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Japan,Morocco,1.0,1.0,1.0,0.0,2.0,0.0,Japan,True
4,77,Round of 32,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Ecuador,Norway,1.0,1.0,2.0,0.0,4.0,0.0,Ecuador,True
5,78,Round of 32,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,Turkey,3.0,1.0,1.0,0.0,1.0,0.0,France,False
6,79,Round of 32,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Côte d'Ivoire,1.0,2.0,4.0,0.0,3.0,0.0,Côte d'Ivoire,False
7,80,Round of 32,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Senegal,1.0,0.0,1.0,0.0,1.0,0.0,England,False
8,81,Round of 32,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Austria,2.0,1.0,1.0,0.0,3.0,0.0,Belgium,False
9,82,Round of 32,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),USA,Bosnia and Herzegovina,2.0,0.0,1.0,0.0,1.0,0.0,USA,False


Round of 16


,match_id,round,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,home_yellow_cards,home_red_cards,away_yellow_cards,away_red_cards,match_winner,penalties
16,89,Round of 16,2026-07-04T17:00:00Z,"NRG Stadium, Houston",Winner Match 73,Winner Match 75,Canada,Germany,1.0,2.0,3.0,0.0,2.0,0.0,Germany,False
17,90,Round of 16,2026-07-04T21:00:00Z,"Lincoln Financial Field, Philadelphia",Winner Match 74,Winner Match 77,Netherlands,Ecuador,1.0,1.0,4.0,0.0,0.0,0.0,Netherlands,True
18,91,Round of 16,2026-07-05T20:00:00Z,"MetLife Stadium, East Rutherford",Winner Match 76,Winner Match 78,Japan,France,1.0,2.0,2.0,0.0,3.0,0.0,France,False
19,92,Round of 16,2026-07-06T00:00:00Z,"Estadio Azteca, Mexico City",Winner Match 79,Winner Match 80,Côte d'Ivoire,England,0.0,2.0,3.0,0.0,1.0,0.0,England,False
20,93,Round of 16,2026-07-06T19:00:00Z,"AT&T Stadium, Dallas",Winner Match 83,Winner Match 84,Colombia,Spain,0.0,2.0,2.0,0.0,2.0,1.0,Spain,False
21,94,Round of 16,2026-07-07T00:00:00Z,"Lumen Field, Seattle",Winner Match 81,Winner Match 82,Belgium,USA,2.0,1.0,1.0,0.0,3.0,0.0,Belgium,False
22,95,Round of 16,2026-07-07T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Match 86,Winner Match 88,Australia,Portugal,0.0,2.0,2.0,0.0,1.0,0.0,Portugal,False
23,96,Round of 16,2026-07-07T20:00:00Z,"BC Place, Vancouver",Winner Match 85,Winner Match 87,Egypt,Argentina,0.0,1.0,0.0,0.0,1.0,0.0,Argentina,False


Quarter-final


...

,match_id,round,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,home_yellow_cards,home_red_cards,away_yellow_cards,away_red_cards,match_winner,penalties
28,101,Semi-final,2026-07-14T19:00:00Z,"AT&T Stadium, Dallas",Winner Match 97,Winner Match 98,Netherlands,Spain,1.0,2.0,1.0,0.0,0.0,0.0,Spain,False
29,102,Semi-final,2026-07-15T19:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Match 99,Winner Match 100,England,Argentina,1.0,1.0,1.0,0.0,1.0,0.0,England,True


Third-place playoff


,match_id,round,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,home_yellow_cards,home_red_cards,away_yellow_cards,away_red_cards,match_winner,penalties
30,103,Third-place playoff,2026-07-18T21:00:00Z,"Hard Rock Stadium, Miami",Loser Match 101,Loser Match 102,Netherlands,Argentina,0.0,2.0,4.0,0.0,1.0,0.0,Argentina,False


Final


,match_id,round,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,home_yellow_cards,home_red_cards,away_yellow_cards,away_red_cards,match_winner,penalties
31,104,Final,2026-07-19T19:00:00Z,"MetLife Stadium, East Rutherford",Winner Match 101,Winner Match 102,Spain,England,2.0,1.0,3.0,0.0,2.0,0.0,Spain,False
